# GitHub API Metadata Inspector

Inspect what `description` and `topics` fields are available from the GitHub REST API.

In [1]:
import sys, json
sys.path.insert(0, '..')
import httpx

USERNAME = 'ridhwanrazaliwork'

print('Inspecting GitHub API for:', USERNAME)

Inspecting GitHub API for: ridhwanrazaliwork


## 1. Repos List Endpoint
`GET /users/{username}/repos` — shows what description + topics look like in the list response.

In [2]:
url = f'https://api.github.com/users/{USERNAME}/repos?per_page=100&type=public&sort=updated'
resp = httpx.get(url, headers={'Accept': 'application/vnd.github.raw+json'})
repos = resp.json()

print(f'Total repos: {len(repos)}')
print()
print('First 3 repos raw structure (keys):')
for r in repos[:3]:
    print(f'  {r["name"]}: {list(r.keys())}')
print()

print('First 3 repos: name, description, topics, updated_at')
for r in repos[:3]:
    print(f'  {r["name"]}')
    print(f'    description: {r.get("description", "(missing)")}')
    print(f'    topics: {r.get("topics", "(missing)")}')
    print(f'    updated_at: {r.get("updated_at", "(missing)")}')

Total repos: 41

First 3 repos raw structure (keys):
  airflow-etl-nasa-api: ['id', 'node_id', 'name', 'full_name', 'private', 'owner', 'html_url', 'description', 'fork', 'url', 'forks_url', 'keys_url', 'collaborators_url', 'teams_url', 'hooks_url', 'issue_events_url', 'events_url', 'assignees_url', 'branches_url', 'tags_url', 'blobs_url', 'git_tags_url', 'git_refs_url', 'trees_url', 'statuses_url', 'languages_url', 'stargazers_url', 'contributors_url', 'subscribers_url', 'subscription_url', 'commits_url', 'git_commits_url', 'comments_url', 'issue_comment_url', 'contents_url', 'compare_url', 'merges_url', 'archive_url', 'downloads_url', 'issues_url', 'pulls_url', 'milestones_url', 'notifications_url', 'labels_url', 'releases_url', 'deployments_url', 'created_at', 'updated_at', 'pushed_at', 'git_url', 'ssh_url', 'clone_url', 'svn_url', 'homepage', 'size', 'stargazers_count', 'watchers_count', 'language', 'has_issues', 'has_projects', 'has_downloads', 'has_wiki', 'has_pages', 'has_discus

## 2. Individual Repo Endpoint
`GET /repos/{owner}/{repo}` — some fields only appear in the single-repo response.

In [3]:
# Compare: repo list vs individual endpoint for the same repo
repo_name = repos[0]['name']
indiv_resp = httpx.get(f'https://api.github.com/repos/{USERNAME}/{repo_name}',
                        headers={'Accept': 'application/vnd.github.raw+json'})
indiv = indiv_resp.json()

print(f'Repo: {repo_name}')
print(f'  List description: {repos[0].get("description")}')
print(f'  Indiv description: {indiv.get("description")}')
print(f'  List topics: {repos[0].get("topics", "N/A")}')
print(f'  Indiv topics: {indiv.get("topics", "N/A")}')

Repo: airflow-etl-nasa-api
  List description: Learning Airflow ETL with NASA API -> local Postgres (Docker) - queryable via DBeaver
  Indiv description: Learning Airflow ETL with NASA API -> local Postgres (Docker) - queryable via DBeaver
  List topics: ['airflow', 'data-engineering', 'data-pipeline', 'docker', 'etl', 'postgresql']
  Indiv topics: ['airflow', 'data-engineering', 'data-pipeline', 'docker', 'etl', 'postgresql']


## 3. Topics Endpoint (if needed)
`GET /repos/{owner}/{repo}/topics` — the dedicated topics endpoint.

In [4]:
topics_resp = httpx.get(
    f'https://api.github.com/repos/{USERNAME}/{repo_name}/topics',
    headers={'Accept': 'application/vnd.github.mercy-preview+json'}
)
print(f'Status: {topics_resp.status_code}')
if topics_resp.status_code == 200:
    print(f'Topics for {repo_name}: {topics_resp.json()}')
else:
    print(f'Error: {topics_resp.text[:200]}')

Status: 200
Topics for airflow-etl-nasa-api: {'names': ['airflow', 'data-pipeline', 'docker', 'etl', 'postgresql', 'data-engineering']}


## 4. Repos Without Description
Check how many repos have empty descriptions.

In [5]:
empty_desc = [r['name'] for r in repos if not r.get('description')]
print(f'Repos with empty description: {len(empty_desc)} / {len(repos)}')
if empty_desc:
    for name in empty_desc[:5]:
        print(f'  - {name}')

Repos with empty description: 4 / 41
  - mbti_prediction_for_career_counseling
  - data-mining-group-project
  - a-pathfinding-algorithm
  - fastapi-docker-gcp-pipeline
